In [3]:
%load_ext autoreload
%autoreload 2

In [27]:
from IPython.display import display, Latex, Math

In [4]:
import torch
from utils.model_base_utils import (
    get_device,
    load_model_and_tokenizer
)
from utils.text_utils import (
        generate_text_basic,
        generate_text_basic_stream,
        generate_text_basic_cache,
        generate_text_basic_stream_cache,
        generate_text_stream_concat
)

In [5]:
from utils.model_eval_utils import (
    get_last_boxed,
    extract_final_candidate,
    normalize_text,
    sympy_parser,
    equality_check,
    split_into_parts,
    grade_answer,
    render_prompt
)

In [6]:
from utils.data_utils import (
    load_math500_test
)

In [8]:
from pathlib import Path
# import torch
 
# from reasoning_from_scratch.ch02 import (
#     get_device
# )
from reasoning_from_scratch.qwen3 import (
    download_qwen3_small,
    Qwen3Tokenizer,
    Qwen3Model,
    QWEN_CONFIG_06_B
)
 
 

In [ ]:
def load_model_and_tokenizer(
    which_model, device, use_compile, local_dir="qwen3"
):
    if which_model == "base":
 
        download_qwen3_small(
            kind="base", tokenizer_only=False, out_dir=local_dir
        )
 
        tokenizer_path = Path(local_dir) / "tokenizer-base.json"
        model_path = Path(local_dir) / "qwen3-0.6B-base.pth"
        tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)
 
    elif which_model == "reasoning":
 
        download_qwen3_small(
            kind="reasoning", tokenizer_only=False, out_dir=local_dir
        )
 
        tokenizer_path = Path(local_dir) / "tokenizer-reasoning.json"
        model_path = Path(local_dir) / "qwen3-0.6B-reasoning.pth"
        tokenizer = Qwen3Tokenizer(
            tokenizer_file_path=tokenizer_path,
            apply_chat_template=True,
            add_generation_prompt=True,
            add_thinking=True,
        )
 
    else:
        raise ValueError(f"Invalid choice: which_model={which_model}")
 
    model = Qwen3Model(QWEN_CONFIG_06_B)
    model.load_state_dict(torch.load(model_path))
 
    model.to(device)
 
    if use_compile:
        torch._dynamo.config.allow_unspec_int_on_nn_module = True
        model = torch.compile(model)
 
    return model, tokenizer

In [2]:
 
 
WHICH_MODEL = "base"
# WHICH_MODEL = "reasoning"

device = get_device()
# device = torch.device("cpu")
 
model, tokenizer = load_model_and_tokenizer(
    which_model=WHICH_MODEL,
    device=device,
    use_compile=False
)

NameError: name 'get_device' is not defined

In [8]:
# defininig math prompt
prompt = (
    r"If $a+b=3$ and $ab=\tfrac{13}{6}$, "
    r"what is the value of $a^2+b^2$?"
)

In [9]:
# converting prompt into tokenIds
input_token_ids_tensor = torch.tensor(
    tokenizer.encode(prompt),
    device=device
).unsqueeze(0) # adding batch dimension
 
all_token_ids = []

# Generate output tokens from the model, one at a time
for token in generate_text_basic_stream_cache(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=2048,
    eos_token_id=tokenizer.eos_token_id
):
    # Remove batch dimension
    token_id = token.squeeze(0)
    decoded_id = tokenizer.decode(token_id.tolist())
    # print token as its generated
    print(
        decoded_id,   
        end="",
        flush=True
    )
    all_token_ids.append(token_id)

# Decode the full generated sequence into text
all_tokens = tokenizer.decode(all_token_ids)
 

 To find the value of \( a^2 + b^2 \) given that \( a + b = 3 \) and \( ab = \frac{13}{6} \), we can use the following algebraic identity:

\[
a^2 + b^2 = (a + b)^2 - 2ab
\]

**Step 1:** Substitute the given values into the equation.

\[
a^2 + b^2 = (3)^2 - 2 \left( \frac{13}{6} \right)
\]

**Step 2:** Calculate \( (3)^2 \).

\[
(3)^2 = 9
\]

**Step 3:** Calculate \( 2 \times \frac{13}{6} \).

\[
2 \times \frac{13}{6} = \frac{26}{6} = \frac{13}{3}
\]

**Step 4:** Subtract the second result from the first.

\[
a^2 + b^2 = 9 - \frac{13}{3}
\]

**Step 5:** Convert 9 to a fraction with a denominator of 3 to perform the subtraction.

\[
9 = \frac{27}{3}
\]

\[
a^2 + b^2 = \frac{27}{3} - \frac{13}{3} = \frac{14}{3}
\]

**Final Answer:**

\[
\boxed{\dfrac{14}{3}}
\]

In [8]:
# display(Latex(all_tokens))

In [10]:
generated_text = generate_text_stream_concat(
    model, tokenizer, prompt, device,
    max_new_tokens=2048,
    verbose=True
)

 To find the value of \( a^2 + b^2 \) given that \( a + b = 3 \) and \( ab = \frac{13}{6} \), we can use the following algebraic identity:

\[
a^2 + b^2 = (a + b)^2 - 2ab
\]

**Step 1:** Substitute the given values into the equation.

\[
a^2 + b^2 = (3)^2 - 2 \left( \frac{13}{6} \right)
\]

**Step 2:** Calculate \( (3)^2 \).

\[
(3)^2 = 9
\]

**Step 3:** Calculate \( 2 \times \frac{13}{6} \).

\[
2 \times \frac{13}{6} = \frac{26}{6} = \frac{13}{3}
\]

**Step 4:** Subtract the second result from the first.

\[
a^2 + b^2 = 9 - \frac{13}{3}
\]

**Step 5:** Convert 9 to a fraction with a denominator of 3 to perform the subtraction.

\[
9 = \frac{27}{3}
\]

\[
a^2 + b^2 = \frac{27}{3} - \frac{13}{3} = \frac{14}{3}
\]

**Final Answer:**

\[
\boxed{\dfrac{14}{3}}
\]

In [ ]:
display(Latex(generated_text))

In [12]:
generated_text

' To find the value of \\( a^2 + b^2 \\) given that \\( a + b = 3 \\) and \\( ab = \\frac{13}{6} \\), we can use the following algebraic identity:\n\n\\[\na^2 + b^2 = (a + b)^2 - 2ab\n\\]\n\n**Step 1:** Substitute the given values into the equation.\n\n\\[\na^2 + b^2 = (3)^2 - 2 \\left( \\frac{13}{6} \\right)\n\\]\n\n**Step 2:** Calculate \\( (3)^2 \\).\n\n\\[\n(3)^2 = 9\n\\]\n\n**Step 3:** Calculate \\( 2 \\times \\frac{13}{6} \\).\n\n\\[\n2 \\times \\frac{13}{6} = \\frac{26}{6} = \\frac{13}{3}\n\\]\n\n**Step 4:** Subtract the second result from the first.\n\n\\[\na^2 + b^2 = 9 - \\frac{13}{3}\n\\]\n\n**Step 5:** Convert 9 to a fraction with a denominator of 3 to perform the subtraction.\n\n\\[\n9 = \\frac{27}{3}\n\\]\n\n\\[\na^2 + b^2 = \\frac{27}{3} - \\frac{13}{3} = \\frac{14}{3}\n\\]\n\n**Final Answer:**\n\n\\[\n\\boxed{\\dfrac{14}{3}}\n\\]'

In [32]:
extracted_answer = get_last_boxed(generated_text)
extracted_answer

'\\dfrac{14}{3}'

In [33]:
display(Math(extracted_answer))

<IPython.core.display.Math object>

In [34]:
extract_final_candidate(generated_text)

'\\dfrac{14}{3}'

In [40]:
print(extract_final_candidate(r"\boxed{ 14/3. }"))

14/3.


In [41]:
print(extract_final_candidate("abc < > 14/3 abc"))

14/3


In [44]:
print(normalize_text(extract_final_candidate(generated_text)))

(14)/(3)


In [47]:
print(normalize_text(r"\text{\[\frac{14}{3}\]}"))

(14)/(3)


In [49]:
import sympy
sympy.__version__

'1.14.0'

In [55]:
print(sympy_parser(normalize_text(
    extract_final_candidate(generated_text)
)))

14/3


In [56]:
print(sympy_parser("28/6"))

14/3


In [61]:
print(equality_check(
    normalize_text("13/4."),
    normalize_text(r"(13)/(4)")
))

True


In [62]:
print(equality_check(
    normalize_text("0.5"),
    normalize_text(r"(1)/(2)")
))

True


In [63]:
print(equality_check(
    normalize_text("14/3"),
    normalize_text("15/3")
))

False


In [64]:
# input with tuples
print(equality_check(
    normalize_text("(14/3, 2/3)"),
    normalize_text("(14/3, 4/6)")
))

False


In [67]:
split_into_parts(normalize_text(r"(14/3, 2/3)"))

['14/3', '2/3']

In [70]:
grade_answer("14/3", r"\frac{14}{3}")

True

In [71]:
grade_answer(r"(14/3, 2/3)", "(14/3, 4/6)")

True

In [73]:
tests = [
        ("check_1", "3/4", r"\frac{3}{4}", True),
        ("check_2", "(3)/(4)", r"3/4", True),
        ("check_3", r"\frac{\sqrt{8}}{2}", "sqrt(2)", True),
        ("check_4", r"\( \frac{1}{2} + \frac{1}{6} \)", "2/3", True),
        ("check_5", "(1, 2)", r"(1,2)", True),
        ("check_6", "(2, 1)", "(1, 2)", False),
        ("check_7", "(1, 2, 3)", "(1, 2)", False),
        ("check_8", "0.5", "1/2", True),
        ("check_9", "0.3333333333", "1/3", False),
        ("check_10", "1,234/2", "617", True),
        ("check_11", r"\text{2/3}", "2/3", True),
        ("check_12", "50%", "1/2", False),
        ("check_13", r"2\cdot 3/4", "3/2", True),
        ("check_14", r"90^\circ", "90", True),
        ("check_15", r"\left(\frac{3}{4}\right)", "3/4", True),
        ("check_16", r"2²", "2**2", True),
    ]
 

In [74]:
 def run_demos_table(tests):
    header = ("Test", "Expect", "Got", "Status")
    rows = []
    
    for name, pred, gtruth, expect in tests:
        # Run equality check
        got = grade_answer(pred, gtruth)
        status = "PASS" if got == expect else "FAIL"
        rows.append((name, str(expect), str(got), status))
 
    data = [header] + rows

    # Compute max width for each column to align table nicely
    col_widths = [
        max(len(row[i]) for row in data)
        for i in range(len(header))
    ]

    # Print table row by row
    for row in data:
        line = " | ".join(
            row[i].ljust(col_widths[i])
            for i in range(len(header))
        )
        print(line)
        
    # Print summary of passed tests
    passed = sum(r[3] == "PASS" for r in rows)
    print(f"\nPassed {passed}/{len(rows)}")

In [76]:
run_demos_table(tests)

Test     | Expect | Got   | Status
check_1  | True   | True  | PASS  
check_2  | True   | True  | PASS  
check_3  | True   | True  | PASS  
check_4  | True   | True  | PASS  
check_5  | True   | True  | PASS  
check_6  | False  | False | PASS  
check_7  | False  | False | PASS  
check_8  | True   | True  | PASS  
check_9  | False  | False | PASS  
check_10 | True   | True  | PASS  
check_11 | True   | True  | PASS  
check_12 | False  | False | PASS  
check_13 | True   | True  | PASS  
check_14 | True   | True  | PASS  
check_15 | True   | True  | PASS  
check_16 | True   | True  | PASS  

Passed 16/16


In [77]:
from datasets import load_dataset

ds = load_dataset("HuggingFaceH4/MATH-500")

README.md:   0%|          | 0.00/412 [00:00<?, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

In [79]:
ds

DatasetDict({
    test: Dataset({
        features: ['problem', 'solution', 'answer', 'subject', 'level', 'unique_id'],
        num_rows: 500
    })
})

In [11]:
math_data = load_math500_test()
print("Number of entries:", len(math_data))

Number of entries: 500


In [12]:
from pprint import pprint
pprint(math_data[0])

{'answer': '\\left( 3, \\frac{\\pi}{2} \\right)',
 'level': 2,
 'problem': 'Convert the point $(0,3)$ in rectangular coordinates to polar '
            'coordinates.  Enter your answer in the form $(r,\\theta),$ where '
            '$r > 0$ and $0 \\le \\theta < 2 \\pi.$',
 'solution': 'We have that $r = \\sqrt{0^2 + 3^2} = 3.$  Also, if we draw the '
             'line connecting the origin and $(0,3),$ this line makes an angle '
             'of $\\frac{\\pi}{2}$ with the positive $x$-axis.\n'
             '\n'
             '[asy]\n'
             'unitsize(0.8 cm);\n'
             '\n'
             'draw((-0.5,0)--(3.5,0));\n'
             'draw((0,-0.5)--(0,3.5));\n'
             'draw(arc((0,0),3,0,90),red,Arrow(6));\n'
             '\n'
             'dot((0,3), red);\n'
             'label("$(0,3)$", (0,3), W);\n'
             'dot((3,0), red);\n'
             '[/asy]\n'
             '\n'
             'Therefore, the polar coordinates are $\\boxed{\\left( 3, '
             '\\frac

In [13]:
prompt = (
    r"If $a+b=3$ and $ab=\tfrac{13}{6}$, "
    r"what is the value of $a^2+b^2$?"
)
prompt_fmt = render_prompt(prompt)
print(prompt_fmt)

You are a helpful math assistant.
Answer the question and write the final result on a new line as:
\boxed{ANSWER}

Problem:
If $a+b=3$ and $ab=\tfrac{13}{6}$, what is the value of $a^2+b^2$?

Answer:


In [14]:
generated_text = generate_text_stream_concat(
    model, tokenizer, prompt_fmt, device,
    max_new_tokens=2048,
    verbose=True
)

 \boxed{10}

In [96]:
generated_text

' \\boxed{10}'

In [15]:
def mini_eval_demo(model, tokenizer, device):
    # Test example with "problem" and "answer" fields
    ex = {
        "problem": "Compute 1/2 + 1/6.",
        "answer": "2/3"
    }
    prompt = render_prompt(ex["problem"])
    gen_text = generate_text_stream_concat(
        model, tokenizer, prompt, device,
        max_new_tokens=64,
    )
    pred_answer = extract_final_candidate(gen_text)
    is_correct = grade_answer(
        pred_answer, ex["answer"]
    )
    
    print(f"Device: {device}")
    print(f"Prediction: {pred_answer}")
    print(f"Ground truth: {ex['answer']}")
    print(f"Correct: {is_correct}")

In [16]:
mini_eval_demo(model, tokenizer, device)

Device: mps
Prediction: 1/3
Ground truth: 2/3
Correct: False


In [17]:
import time
import json
 
def eta_progress_message(
    processed,
    total,
    start_time,
    show_eta=False,
    label="Progress",
):
    progress = f"{label}: {processed}/{total}"
    if not show_eta or processed <= 0:          
        return progress
 
    elapsed = time.time() - start_time
    if elapsed <= 0:
        return progress
 
    remaining = max(total - processed, 0)
 
    if processed:
        avg_time = elapsed / processed
        eta_seconds = avg_time * remaining
    else:
        eta_seconds = 0
 
    eta_seconds = max(int(round(eta_seconds)), 0)
    minutes, rem_seconds = divmod(eta_seconds, 60)
    hours, minutes = divmod(minutes, 60)
    if hours:
        eta = f"{hours}h {minutes:02d}m {rem_seconds:02d}s"
    elif minutes:
        eta = f"{minutes:02d}m {rem_seconds:02d}s"
    else:
        eta = f"{rem_seconds:02d}s"
 
    return f"{progress} | ETA: {eta}"
 
 
def evaluate_math500_stream(
    model,
    tokenizer,
    device,
    math_data,
    out_path=None,
    max_new_tokens=512,
    verbose=False,
):
 
    if out_path is None:
        dev_name = str(device).replace(":", "-") # Make filename compatible with Windows
        out_path = Path(f"eval/math500_{WHICH_MODEL}-{dev_name}.jsonl")
 
    num_examples = len(math_data)
    num_correct = 0
    start_time = time.time()
 
    with open(out_path, "w", encoding="utf-8") as f: #Save results for inspection
        for i, row in enumerate(math_data, start=1):
            prompt = render_prompt(row["problem"])
            gen_text = generate_text_stream_concat(
                model, tokenizer, prompt, device,
                max_new_tokens=max_new_tokens,
                verbose=verbose,
            )
 
            extracted = extract_final_candidate(
                gen_text
            )
            is_correct = grade_answer(
                extracted, row["answer"]
            )
            num_correct += int(is_correct)
 
            record = {
                "index": i,
                "problem": row["problem"],
                "gtruth_answer": row["answer"],
                "generated_text": gen_text,
                "extracted": extracted,
                "correct": bool(is_correct),
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
 
            progress_msg = eta_progress_message(
                processed=i,
                total=num_examples,
                start_time=start_time,
                show_eta=True,
                label="MATH-500",
            )
            print(progress_msg, end="\r", flush=True)
 
            if verbose:
                print(
                    f"\n\n{'='*50}\n{progress_msg}\n"
                    f"{'='*50}\nExtracted: {extracted}\n"
                    f"Expected:  {row['answer']}\n"
                    f"Correct so far: {num_correct}\n{'-'*50}"
                )
 
    seconds_elapsed = time.time() - start_time
    acc = num_correct / num_examples if num_examples else 0.0
    print(f"\nAccuracy: {acc*100:.1f}% ({num_correct}/{num_examples})")
    print(f"Total time: {seconds_elapsed/60:.1f} min")
    print(f"Logs written to: {out_path}")
    return num_correct, num_examples, acc

In [18]:
print("Model:", WHICH_MODEL)
num_correct, num_examples, acc = evaluate_math500_stream(
    model, tokenizer, device, 
    math_data=math_data[:10],
    max_new_tokens=2048,
    verbose=False
)

Model: base
MATH-500: 10/10 | ETA: 00s
Accuracy: 40.0% (4/10)
Total time: 0.4 min
Logs written to: math500_base-mps.jsonl


In [10]:
device = "mps"
dev_name = str(device).replace(":", "-")
local_path = Path(f"eval/math500_{WHICH_MODEL}-{dev_name}.jsonl")
local_path
# if not local_path.exists():
#     raise FileNotFoundError(
#         f"{local_path} not found."
#     )

# results = []
# total_generated_text_len = 0
# i = 0
# with open(local_path, "r") as f:
#     for line in f:
#         i += 1
#         if line.strip():
#             line_json = json.loads(line)
#             results.append(line_json)
#             total_generated_text_len += len(tokenizer.encode(line_json["generated_text"]))
#     avg_response_len = total_generated_text_len/i
#     print(f"average response length: {avg_response_len:2f} tokens")

PosixPath('eval/math500_base-mps.jsonl')

In [125]:
results[0]

{'index': 1,
 'problem': 'Convert the point $(0,3)$ in rectangular coordinates to polar coordinates.  Enter your answer in the form $(r,\\theta),$ where $r > 0$ and $0 \\le \\theta < 2 \\pi.$',
 'gtruth_answer': '\\left( 3, \\frac{\\pi}{2} \\right)',
 'generated_text': ' $\\boxed{(3,\\frac{\\pi}{2})}$.',
 'extracted': '(3,\\frac{\\pi}{2})',
 'correct': True}